### error_back_propagation_method
**误差反向传播法**  
上一章中，我们介绍了神经网络的学习，并通过数值微分计算了神经网络的权重参数的梯度（严格来说，是损失函数关于权重参数的梯度）。数值微分虽然简单，也容易实现，但缺点是计算上比较费时间。本章我们将学习一个能够**高效计算权重参数的梯度的方法**——误差反向传播法。<br> 
要正确理解误差反向传播法，我个人认为有两种方法：一种是基于数学式；另一种是基于计算图（computational graph）。前者是比较常见的方法，机器学习相关的图书中多数都是以数学式为中心展开论述的。因为这种方法严密且简洁，所以确实非常合理，但如果一上来就围绕数学式进行探讨，会忽略 一些根本的东西，止步于式子的罗列。因此，本章希望大家通过计算图，直 观地理解误差反向传播法。然后，再结合实际的代码加深理解，相信大家一 定会有种“原来如此！”的感觉。<br>
此外，通过计算图来理解误差反向传播法这个想法，参考了Andrej Karpathy的博客[4]和他与Fei-Fei Li教授负责的斯坦福大学的深度学习课程 CS231n [5] 。

#### 计算图
计算图将计算过程用图形表示出来。这里说的图形是数据结构图，通过多个节点和边表示（连接节点的直线称为“边”）。为了让大家熟悉计算图，本节先用计算图解一些简单的问题。从这些简单的问题开始，逐步深入，最终抵达误差反向传播法。
##### 用计算图求解
现在，我们尝试用计算图解简单的问题。下面我们要看的几个问题都是用心算就能解开的简单问题，这里的目的只是通过它们让大家熟悉计算图。 掌握了计算图的使用方法之后，在后面即将看到的复杂计算中它将发挥巨大威力，所以本节请一定学会计算图的使用方法。

* 问题1： 太郎在超市买了2个100日元一个的苹果，消费税是10%，请计算支付金额。

计算图通过节点和箭头表示计算过程。节点用○表示，○中是计算的内容。将计算的中间结果写在箭头的上方，表示各个节点的计算结果从左向右传递。用计算图解问题1，求解过程如图5-1所示。

<img src="data/error_back_propagation_method/image1.png" style="width:50%;">

如图5-1所示，开始时，苹果的100日元流到“×2”节点，变成200日元，然后被传递给下一个节点。接着，这个200日元流向“×1.1”节点，变成220 日元。因此，从这个计算图的结果可知，答案为220日元。<br>

<img src="data/error_back_propagation_method/image2.png" style="width:50%;">

虽然图5-1中把“×2” “×1.1”等作为一个运算整体用○括起来了，不过只用○表示乘法运算“×”也是可行的。此时，如图5-2所示，可以将“2”和 “1.1”分别作为变量“苹果的个数”和“消费税”标在○外面。

* 问题2：太郎在超市买了2个苹果、3个橘子。其中，苹果每个100日元，橘子每个150日元。消费税是10%，请计算支付金额。

<img src="data/error_back_propagation_method/image3.png" style="width:50%;">

这个问题中新增了加法节点“+”，用来合计苹果和橘子的金额。构建了计算图后，从左向右进行计算。就像电路中的电流流动一样，计算结果从左 向右传递。到达最右边的计算结果后，计算过程就结束了。从图5-3中可知，问题2的答案为715日元。

综上，用计算图解题的情况下，需要按如下流程进行。 
1. 构建计算图。 
2. 在计算图上，从左向右进行计算。 

这里的第2歩“从左向右进行计算”是一种正方向上的传播，简称为正向传播（forward propagation）。**正向传播是从计算图出发点到结束点的传播。**既然有正向传播这个名称，当然也可以考虑反向（从图上看的话，就是从右向左）的传播。实际上，这种传播称为反向传播（backward propagation）。**反向传播将在接下来的导数计算中发挥重要作用**。


##### 局部计算
计算图的特征是可以通过传递“局部计算”获得最终结果。“局部”这个词的意思是“与自己相关的某个小范围”。局部计算是指，无论全局发生了什么，都能只根据与自己相关的信息输出接下来的结果。<br>
我们用一个具体的例子来说明局部计算。比如，在超市买了2个苹果和其他很多东西。此时，可以画出如图5-4所示的计算图。<br>

<img src="data/error_back_propagation_method/image4.png" style="width:50%;">

如图5-4所示，假设（经过复杂的计算）购买的其他很多东西总共花费4000日元。这里的重点是，各个节点处的计算都是局部计算。这意味着，例 如苹果和其他很多东西的求和运算（4000 + 200→4200）并不关心4000这个 数字是如何计算而来的，只要把两个数字相加就可以了。换言之，各个节点处只需进行与自己有关的计算（在这个例子中是对输入的两个数字进行加法运算），不用考虑全局。<br>
综上，计算图可以集中精力于局部计算。**无论全局的计算有多么复杂，各个步骤所要做的就是对象节点的局部计算**。虽然局部计算非常简单，但是 通过传递它的计算结果，可以获得全局的复杂计算的结果。

> 比如，组装汽车是一个复杂的工作，通常需要进行“流水线”作业。每个工人（机器）所承担的都是被简化了的工作，这个工作的成果会传递给下一个工人，直至汽车组装完成。计算图将复杂的计算分割成简单的局部计算，和流水线作业一样，将局部计算的结果传递给下一个节点。在将复杂的计算分解成简单的计算这一点上与汽车的组装有相似之处。


##### 为什么用计算图解题
前面我们用计算图解答了两个问题，那么计算图到底有什么优点呢？一个优点就在于前面所说的局部计算。无论全局是多么复杂的计算，都可以通 过局部计算使各个节点致力于简单的计算，从而简化问题。另一个优点是，利用计算图可以将中间的计算结果全部保存起来（比如，计算进行到2个苹果时的金额是200日元、加上消费税之前的金额650日元等）。但是只有这些理由可能还无法令人信服。实际上，使用计算图最大的原因是，可以**通过反向传播高效计算导数**。<br>
在介绍计算图的反向传播时，我们再来思考一下问题1。问题1中，我们计算了购买2个苹果时加上消费税最终需要支付的金额。这里，假设我们想知道苹果价格的上涨会在多大程度上影响最终的支付金额，即求“支付金额关于苹果的价格的导数”。设苹果的价格为x，支付金额为L，则相当于求 $\frac{\partial L}{\partial x}$ 。这个导数的值表示当苹果的价格稍微上涨时，支付金额会增加多少。<br>
如前所述，“支付金额关于苹果的价格的导数”的值可以通过计算图的反向传播求出来。先来看一下结果，如图5-5所示，可以通过计算图的反向 传播求导数（关于如何进行反向传播，接下来马上会介绍）。

<img src="data/error_back_propagation_method/image5.png" style="width:50%;">

如图5-5所示，反向传播使用与正方向相反的箭头（粗线）表示。反向传播传递“局部导数”，将导数的值写在箭头的下方。在这个例子中，反向传 播从右向左传递导数的值（1→1.1→2.2）。从这个结果中可知，“支付金额关于苹果的价格的导数”的值是2.2。这意味着，**如果苹果的价格上涨1日元，最终的支付金额会增加2.2日元**（严格地讲，如果苹果的价格增加某个微小值，则最终的支付金额将增加那个微小值的2.2倍）。<br>
这里只求了关于苹果的价格的导数，不过“支付金额关于消费税的导数” “支付金额关于苹果的个数的导数”等也都可以用同样的方式算出来。并 且，**计算中途求得的导数的结果（中间传递的导数）可以被共享，从而可以高效地计算多个导数**。综上，计算图的优点是，可以通**过正向传播和反向传播高效地计算各个变量的导数值。**


#### 链式法则
前面介绍的计算图的正向传播将计算结果正向（从左到右）传递，其计算过程是我们日常接触的计算过程，所以感觉上可能比较自然。而反向传播将局部导数向正方向的反方向（从右到左）传递，一开始可能会让人感到困惑。传递这个局部导数的原理，是基于链式法则（chain rule）的。本节将介绍链式法则，并阐明它是如何对应计算图上的反向传播的。

##### 计算图的反向传播
话不多说，让我们先来看一个使用计算图的反向传播的例子。假设存在 y = f(x) 的计算，这个计算的反向传播如图5-6所示。

<img src="data/error_back_propagation_method/image6.png" style="width:50%;">

如图所示，反向传播的计算顺序是，将信号E乘以节点的局部导数（$\frac{\partial y}{\partial x}$），然后将结果传递给下一个节点。**这里所说的局部导数是指正向传播 中 y = f(x) 的导数**，也就是 y 关于 x 的导数（$\frac{\partial y}{\partial x}$）。比如，假设 $y = f(x) = x^2 $，则局部导数为 $\frac{\partial y}{\partial x} = 2x$。把这个局部导数乘以上游传过来的值（本例中为E），然后传递给前面的节点。<br>
这就是反向传播的计算顺序。通过这样的计算，可以高效地求出导数的值，这是反向传播的要点。那么这是如何实现的呢？我们可以从链式法则的 原理进行解释。下面我们就来介绍链式法则。

##### 什么是链式法则
介绍链式法则时，我们需要先从复合函数说起。复合函数是由多个函数构成的函数。比如，$z = (x + y)^2$是由式（5.1）所示的两个式子构成的。

$\begin{array}{c}z=t^{2} \\t=x+y\end{array}   \quad \quad(5.1)$

链式法则是关于复合函数的导数的性质，定义如下。
> 如果某个函数由复合函数表示，则该复合函数的导数可以用构成复合函数的各个函数的导数的乘积表示。

这就是链式法则的原理，乍一看可能比较难理解，但实际上它是一个非常简单的性质。以式（5.1）为例， （z关于x的导数）可以用 （z关于t 的导数）和 （t关于x的导数）的乘积表示。用数学式表示的话，可以写成 式（5.2）。